# 🚀 Enterprise Staged Sentiment Analysis Training Pipeline
### End-to-End GPU Training: Stage A (Domain Adaptation) → Stage B (3-Class Fine-Tuning) → Stage C (Strict Evaluation)

This notebook trains an enterprise-grade social media sentiment analysis model (`negative`, `neutral`, `positive`) using transfer learning:
1. **Stage A (Domain Adaptation):** Fine-tunes `google-bert/bert-base-uncased` on **Sentiment140** (1.23M train, 154K val, 154K test) with enterprise-grade text cleaning (HTML entity decoding, URL stripping, `@user` standardization, and character elongation squashing). Evaluates on an independent 10% test set to verify zero underfitting/overfitting.
2. **Stage B (Three-Way Fine-Tuning):** Swaps the classification head to 3 classes (`negative`, `neutral`, `positive`) and fine-tunes on CardiffNLP TweetEval (45.6K train, 2K val, 12.3K test).
3. **Stage C (Evaluation & Diagnostics):** Generates confusion matrices, per-class F1/precision/recall tables, an interactive real-time prediction playground, and exports a production-ready model bundle to Google Drive.

> ⚠️ **Hardware Requirement:** Ensure GPU accelerator is enabled: **Runtime → Change runtime type → T4 GPU (or A100)**.

## 1. Hardware & Environment Diagnostics

In [ ]:
import torch

print("=" * 60)
print("PYTORCH & CUDA DIAGNOSTICS")
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Model:       {gpu_name}")
    print(f"VRAM Capacity:   {vram_gb:.2f} GB")
    print(f"Device Index:    {torch.cuda.current_device()}")
else:
    print("⚠️ WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")
print("=" * 60)

!nvidia-smi

## 2. Sync Repository & Install Dependencies

In [ ]:
import os

# Clone repository if not already cloned, or pull latest changes
REPO_URL = "https://github.com/HabtamuShewamene/AIproject.git"
BRANCH = "phase/2-data-model-audit"

if not os.path.exists("AIproject"):
    print("Cloning repository...")
    !git clone $REPO_URL
    %cd AIproject
    !git checkout $BRANCH
else:
    print("Repository already exists. Pulling latest commits...")
    %cd AIproject
    !git checkout $BRANCH
    !git pull origin $BRANCH

# Install core training and evaluation libraries
!pip install -q transformers>=4.46.0 datasets>=3.1.0 accelerate>=1.1.0 scikit-learn pandas seaborn matplotlib

## 3. Supply Sentiment140 Dataset
Smart multi-source discovery: detects local drag-and-drop, searches Google Drive, or automatically downloads directly from the official Stanford mirror in 15 seconds.

In [ ]:
import os
import shutil
from pathlib import Path

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
RAW_TARGET = RAW_DIR / "training.1600000.processed.noemoticon.csv"

# Check 1: Already uploaded or present locally
if RAW_TARGET.is_file() and RAW_TARGET.stat().st_size > 100_000_000:
    print(f"✅ Dataset already present in {RAW_TARGET} ({RAW_TARGET.stat().st_size / (1024**2):.1f} MB)")
else:
    found = False
    # Check 2: Google Drive (if mounted)
    if os.path.exists('/content/drive/MyDrive'):
        drive_root = Path('/content/drive/MyDrive')
        candidates = list(drive_root.rglob("training.1600000.processed.noemoticon.csv"))
        if candidates:
            print(f"Found on Google Drive: {candidates[0]}")
            shutil.copy2(candidates[0], RAW_TARGET)
            print(f"✅ Copied from Google Drive ({RAW_TARGET.stat().st_size / (1024**2):.1f} MB)")
            found = True
            
    # Check 3: Automated high-speed download directly from Stanford repository (~15 seconds)
    if not found and not RAW_TARGET.is_file():
        print("No local/Drive file found. Automatically downloading Sentiment140 archive directly into Colab (~15s)...")
        !wget -q --show-progress -O data/raw/trainingandtestdata.zip http://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip
        !unzip -q -o data/raw/trainingandtestdata.zip -d data/raw/
        if RAW_TARGET.is_file():
            print(f"✅ Successfully downloaded & unpacked {RAW_TARGET.name} ({RAW_TARGET.stat().st_size / (1024**2):.1f} MB)")
        else:
            print("⚠️ Download completed, please check data/raw/ or drag & drop your local file into the left file sidebar.")

## 4. Run Enterprise Preprocessing & 80/10/10 Splitting
Applies HTML unescaping, URL removal, mention standardization (`@user`), hashtag conversion, elongation compression, noise filtering, and deterministic cryptographic 80/10/10 splitting.

In [ ]:
# Execute enterprise Sentiment140 preparation (80% train, 10% val, 10% test)
!python -m src.prepare_sentiment140

# Download and validate pinned TweetEval 3-class benchmark splits
!python -m src.download_tweet_eval

# Display generated files and sizes
!ls -lh data/processed/

## 5. Stage A: Binary Social Domain Adaptation
Fine-tunes `google-bert/bert-base-uncased` on the enterprise-cleaned Sentiment140 corpus.
* **Default:** Trains on full 80% train split (~1.23M rows) for 2 epochs.
* *Optional:* Add `--max-samples 250000` if you want a faster run.

In [ ]:
# Run Stage A (Sentiment140 Binary Adaptation)
# Automatically saves best model to models/stage_a_sentiment140
# and evaluates on the independent 10% test set (154K rows)
!python -m src.train_staged_sentiment --stage a --epochs 2

## 6. Stage B: Three-Class TweetEval Fine-Tuning
Loads the adapted encoder from Stage A, replaces the classification head with 3 classes (`negative`, `neutral`, `positive`), and fine-tunes on TweetEval.

In [ ]:
# Run Stage B (Three-Class Fine-Tuning on TweetEval)
# Evaluates on untouched TweetEval test split (12,284 rows)
!python -m src.train_staged_sentiment --stage b --epochs 2

## 7. Stage C: Strict Benchmark Evaluation & Visualization
Inspect the final classification report and render the confusion matrix.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

metrics_file = "results/staged_training_metrics.json"
with open(metrics_file, "r", encoding="utf-8") as f:
    metrics = json.load(f)

report = metrics.get("tweet_eval_test", {})
print("=" * 60)
print("TWEETEVAL 3-CLASS TEST EVALUATION METRICS")
print("=" * 60)

classes = ["negative", "neutral", "positive"]
rows = []
for c in classes:
    if c in report:
        rows.append({
            "Class": c.capitalize(),
            "Precision": f"{report[c]['precision'] * 100:.2f}%",
            "Recall": f"{report[c]['recall'] * 100:.2f}%",
            "F1-Score": f"{report[c]['f1-score'] * 100:.2f}%",
            "Support": report[c]['support'],
        })

df_report = pd.DataFrame(rows)
display(df_report)

macro_f1 = report.get("macro avg", {}).get("f1-score", 0)
accuracy = report.get("accuracy", 0)
print(f"\nOverall Test Accuracy: {accuracy * 100:.2f}%")
print(f"Macro Average F1:     {macro_f1 * 100:.2f}%")
print("=" * 60)

## 8. Interactive Real-Time Prediction Playground
Test your fine-tuned 3-class sentiment model on custom text, edge cases, sarcasm, and neutral statements.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "models/bert_sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

id2label = {0: "negative", 1: "neutral", 2: "positive"}

def analyze_sentiment(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = int(torch.argmax(probs).item())
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id].item())
    
    print(f"Input Text:   '{text}'")
    print(f"Sentiment:    {pred_label.upper()} ({confidence * 100:.2f}% confidence)")
    print("Probabilities:")
    for i, label in id2label.items():
        bar = "█" * int(probs[i].item() * 25)
        print(f"  {label.capitalize():8}: {probs[i].item() * 100:5.1f}% | {bar}")
    print("-" * 50)

# Run test examples
analyze_sentiment("I waited 3 hours and nobody helped me. Terrible customer service.")
analyze_sentiment("The meeting was rescheduled to 3:00 PM tomorrow.")
analyze_sentiment("I absolutely love this new update! Everything runs so smoothly.")

## 9. Package & Export Model to Google Drive
Verifies model weights and compresses `models/bert_sentiment/` into a `.zip` archive on Google Drive for local API serving.

In [ ]:
from datetime import datetime
from pathlib import Path

MODEL_PATH = Path("models/bert_sentiment")
required_files = ["config.json", "tokenizer.json", "vocab.txt"]
assert MODEL_PATH.is_dir(), "Error: models/bert_sentiment does not exist!"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"bert_sentiment_trained_{timestamp}.zip"

print(f"Creating archive: {zip_name}...")
!zip -r $zip_name models/bert_sentiment/ results/staged_training_metrics.json

# Export to Google Drive
dest_path = f"/content/drive/MyDrive/{zip_name}"
!cp $zip_name "$dest_path"

print("=" * 60)
print(f"🎉 MODEL EXPORTED SUCCESSFULLY!")
print(f"Saved in Google Drive: {dest_path}")
print("You can download this zip and extract it into your local 'models/bert_sentiment/' directory.")
print("=" * 60)